# 06 — ACORN (Educational Reproduction)

hnswlib gives no access to its internal per-node neighbor lists or construction hooks, so **true ACORN-gamma cannot be built on top of hnswlib**. This notebook implements two honest, clearly-distinguished approximations rather than pretending either one is the paper's algorithm.

This notebook is self-contained: it installs its own deps, loads data, builds embeddings, and writes its own results CSV. Only the methodology, config constants, seed, and corpus/query construction are shared verbatim across all 9 notebooks in this study.


## 1. Install

In [1]:
# Core install cell. Safe to re-run. Each notebook is independently runnable.
!pip install -q datasets sentence-transformers hnswlib scikit-learn pandas numpy tqdm


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Imports

In [2]:
import os, time, json, math
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

np.set_printoptions(suppress=True)
pd.set_option("display.max_columns", 50)

import hnswlib


## 3. Shared configuration

In [3]:
# ============================================================
# SHARED CONFIGURATION — identical across all 9 notebooks.
# Only DEBUG_MODE changes corpus/query size; everything else fixed.
# ============================================================
DEBUG_MODE   = True     # small N for correctness checks; set False for full run

N_CORPUS     = 60_000 if not DEBUG_MODE else 3_000
N_QUERIES    = 1_000  if not DEBUG_MODE else 100
K            = 10
SEED         = 42

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # 384-dim
HNSW_M               = 16
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH       = 100
HNSW_SPACE           = "cosine"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG_MODE={DEBUG_MODE}  N_CORPUS={N_CORPUS}  N_QUERIES={N_QUERIES}  DEVICE={DEVICE}")


DEBUG_MODE=True  N_CORPUS=3000  N_QUERIES=100  DEVICE=cuda


## 4. Dataset — AG News corpus/query construction (shared, verbatim)

**Synthetic attributes, labeled explicitly:** `synthetic_tag` (uniform(0,1) per corpus doc) is used to carve the 10%/5% sub-selectivity filters out of a single-label 25% filter. `synthetic_range_attr` (uniform int in [0,10000)) has no semantic relationship to the article text and exists only so Notebook 08 can run a range-filtered experiment (AG News's label is categorical, not a range attribute).

In [4]:
# ============================================================
# DATASET — AG News. Hard-coded facts per spec:
#   4 classes, exactly balanced: 0=World, 1=Sports, 2=Business, 3=Sci/Tech
#   train: 120,000 rows (30,000/class). test: 7,600 rows (1,900/class).
# Corpus is drawn from train (subsampled to N_CORPUS), queries from test
# (subsampled to N_QUERIES), using SEED=42 for the subsample. This is why
# natural single-label filters have ~25% selectivity — not an arbitrary
# number, it follows directly from AG News's exact 4-way class balance.
# ============================================================
ds = load_dataset("fancyzhx/ag_news")
assert ds["train"].num_rows == 120_000
assert ds["test"].num_rows == 7_600

rng = np.random.default_rng(SEED)

train_idx_all = np.arange(ds["train"].num_rows)
rng_perm = np.random.default_rng(SEED)
corpus_idx = rng_perm.choice(train_idx_all, size=min(N_CORPUS, len(train_idx_all)), replace=False)
corpus_idx.sort()

test_idx_all = np.arange(ds["test"].num_rows)
rng_perm2 = np.random.default_rng(SEED)
query_idx = rng_perm2.choice(test_idx_all, size=min(N_QUERIES, len(test_idx_all)), replace=False)
query_idx.sort()

corpus_texts  = [ds["train"][int(i)]["text"]  for i in corpus_idx]
corpus_labels = np.array([ds["train"][int(i)]["label"] for i in corpus_idx], dtype=np.int64)

query_texts   = [ds["test"][int(i)]["text"]   for i in query_idx]
query_labels  = np.array([ds["test"][int(i)]["label"] for i in query_idx], dtype=np.int64)

# ------------------------------------------------------------
# SYNTHETIC ATTRIBUTES (explicitly labeled as synthetic; not AG News data).
# synthetic_tag: per-corpus-doc uniform(0,1), used to carve the 10%/5%
#   "synthetic controlled" selectivity filters out of a 25% single-label
#   filter (label==c AND synthetic_tag<0.4 / <0.2).
# synthetic_range_attr: per-corpus-doc uniform integer in [0, 10000), used
#   ONLY by Notebook 08 (UNIFY) to construct a range-filtered ANN
#   experiment, since AG News's label is categorical, not a range
#   attribute. It has NO semantic relationship to the article text —
#   it exists purely to make a range-filtered benchmark possible, mirroring
#   the UNIFY paper's own methodology for attribute-less datasets
#   (SIFT1M/GIST1M: uniform random value in [0, 10000)).
# Both are assigned once here, with a fixed seed, and reused identically
# across every notebook that needs them.
# ------------------------------------------------------------
tag_rng = np.random.default_rng(SEED)
synthetic_tag = tag_rng.uniform(0.0, 1.0, size=len(corpus_idx))

range_rng = np.random.default_rng(SEED)
synthetic_range_attr = range_rng.integers(0, 10_000, size=len(corpus_idx))

corpus_df = pd.DataFrame({
    "corpus_pos": np.arange(len(corpus_idx)),
    "train_idx": corpus_idx,
    "label": corpus_labels,
    "synthetic_tag": synthetic_tag,
    "synthetic_range_attr": synthetic_range_attr,
})
print(corpus_df["label"].value_counts().sort_index())
print(corpus_df.head())


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

label
0    768
1    767
2    704
3    761
Name: count, dtype: int64
   corpus_pos  train_idx  label  synthetic_tag  synthetic_range_attr
0           0         61      2       0.773956                   892
1           1        125      3       0.438878                  7739
2           2        146      3       0.858598                  6545
3           3        182      3       0.697368                  4388
4           4        196      3       0.094177                  4330


## 5. Embedding generation

In [5]:
# ============================================================
# EMBEDDING GENERATION — generated once per notebook, cached in-memory
# (and to /content/*.npy for reuse within the same runtime). Never assume
# a prior notebook already produced these files.
# ============================================================
_model = SentenceTransformer(EMBED_MODEL, device=DEVICE)

def embed(texts, cache_path):
    if os.path.exists(cache_path):
        arr = np.load(cache_path)
        if arr.shape[0] == len(texts):
            return arr
    t0 = time.time()
    arr = _model.encode(
        texts, batch_size=128, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    print(f"Embedded {len(texts)} texts in {time.time()-t0:.2f}s")
    np.save(cache_path, arr)
    return arr

corpus_emb = embed(corpus_texts, "/content/corpus_emb.npy" if os.path.isdir("/content") else "corpus_emb.npy")
query_emb  = embed(query_texts,  "/content/query_emb.npy"  if os.path.isdir("/content") else "query_emb.npy")
print(corpus_emb.shape, query_emb.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Embedded 3000 texts in 4.73s


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 100 texts in 0.27s
(3000, 384) (100, 384)


## 6. Filter design (§4, shared)

In [6]:
# ============================================================
# FILTER DESIGN (per spec §4) — the concrete, non-ambiguous scheme every
# notebook uses. "synthetic controlled" filters explicitly use
# synthetic_tag (see markdown above); everything else is a natural label
# filter derived from AG News's exact class balance.
# ============================================================
def build_filters(corpus_df):
    """Return dict: filter_name -> (selectivity_target_pct, boolean mask, type)."""
    labels = corpus_df["label"].values
    tag = corpus_df["synthetic_tag"].values
    filters = {}
    filters["sel100_unfiltered"] = (100, np.ones(len(labels), dtype=bool), "natural")
    filters["sel75_label_ne_3"]  = (75,  labels != 3, "natural")
    filters["sel50_label_in_01"] = (50,  np.isin(labels, [0, 1]), "natural")
    # single-class filters (~25% target) — report all 4 classes' actual results,
    # but use class 0 as the canonical "sel25" filter referenced elsewhere.
    for c in range(4):
        filters[f"sel25_label_eq_{c}"] = (25, labels == c, "natural")
    # synthetic controlled sub-selectivity filters, built ON TOP OF the
    # canonical single-class filter (class 0), per spec.
    filters["sel10_label0_tag_lt_0.4"] = (10, (labels == 0) & (tag < 0.4), "synthetic_controlled")
    filters["sel5_label0_tag_lt_0.2"]  = (5,  (labels == 0) & (tag < 0.2), "synthetic_controlled")
    return filters

FILTERS = build_filters(corpus_df)
filter_summary = []
for name, (target_pct, mask, ftype) in FILTERS.items():
    filter_summary.append({
        "filter": name, "target_selectivity_pct": target_pct,
        "total_corpus_size": len(mask),
        "valid_document_count": int(mask.sum()),
        "actual_selectivity_pct": round(100.0 * mask.sum() / len(mask), 3),
        "type": ftype,
    })
filter_summary_df = pd.DataFrame(filter_summary)
print(filter_summary_df)


                    filter  target_selectivity_pct  total_corpus_size  \
0        sel100_unfiltered                     100               3000   
1         sel75_label_ne_3                      75               3000   
2        sel50_label_in_01                      50               3000   
3         sel25_label_eq_0                      25               3000   
4         sel25_label_eq_1                      25               3000   
5         sel25_label_eq_2                      25               3000   
6         sel25_label_eq_3                      25               3000   
7  sel10_label0_tag_lt_0.4                      10               3000   
8   sel5_label0_tag_lt_0.2                       5               3000   

   valid_document_count  actual_selectivity_pct                  type  
0                  3000                 100.000               natural  
1                  2239                  74.633               natural  
2                  1535                  51.167      

## 7. Ground truth + recall (shared)

In [7]:
# ============================================================
# GROUND TRUTH + RECALL — exact cosine top-K over the filter-valid subset,
# computed identically in every notebook that needs it.
# ============================================================
def exact_topk_filtered(query_vecs, corpus_vecs, mask, k):
    """Exact cosine top-k (corpus_vecs assumed L2-normalized) restricted to
    corpus rows where mask is True. Returns (indices, sims), indices are
    positions into the FULL corpus (not the masked subset)."""
    valid_pos = np.nonzero(mask)[0]
    if len(valid_pos) == 0:
        return np.full((len(query_vecs), k), -1, dtype=np.int64), np.zeros((len(query_vecs), k))
    sub = corpus_vecs[valid_pos]                       # (m, d)
    sims = query_vecs @ sub.T                           # (nq, m) cosine sim (both normalized)
    kk = min(k, sub.shape[0])
    top_local = np.argsort(-sims, axis=1)[:, :kk]
    top_global = valid_pos[top_local]
    if kk < k:
        pad_idx = np.full((len(query_vecs), k - kk), -1, dtype=np.int64)
        top_global = np.concatenate([top_global, pad_idx], axis=1)
    top_sims = np.take_along_axis(sims, top_local, axis=1)
    if kk < k:
        top_sims = np.concatenate([top_sims, np.zeros((len(query_vecs), k - kk))], axis=1)
    return top_global, top_sims

def recall_at_k(retrieved_ids, gt_ids, k=K):
    """retrieved_ids, gt_ids: (n_queries, k) int arrays of corpus positions (-1 = missing)."""
    total = 0.0
    for r, g in zip(retrieved_ids, gt_ids):
        gt_set = set(int(x) for x in g if x >= 0)
        if len(gt_set) == 0:
            total += 1.0  # nothing to find, nothing missed
            continue
        r_set = set(int(x) for x in r if x >= 0)
        total += len(r_set & gt_set) / min(k, len(gt_set)) if len(gt_set) < k else len(r_set & gt_set) / k
    return total / len(retrieved_ids)


## 8. Latency / QPS / results utilities (shared)

In [8]:
# ============================================================
# LATENCY / QPS UTILITIES — query execution time only. Install / embedding /
# preprocessing time is reported separately, never folded into query latency.
# ============================================================
def time_queries(fn, n_repeats=1):
    """fn() runs ALL queries once and returns per-query latencies (list of floats, seconds).
    Caller is responsible for making fn() do only the search, not setup."""
    all_lat = []
    for _ in range(n_repeats):
        lat = fn()
        all_lat.extend(lat)
    arr = np.array(all_lat)
    return {
        "mean_latency_ms": float(arr.mean() * 1000),
        "p50_latency_ms": float(np.percentile(arr, 50) * 1000),
        "p95_latency_ms": float(np.percentile(arr, 95) * 1000),
        "qps": float(1.0 / arr.mean()) if arr.mean() > 0 else float("inf"),
    }

def append_result(rows, method, filter_name, selectivity, recall, lat_stats,
                   index_build_time_s, **extra):
    row = {
        "method": method, "filter": filter_name, "selectivity": selectivity,
        "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": index_build_time_s,
    }
    row.update(extra)
    rows.append(row)
    return rows

def save_results(rows, method_slug):
    df = pd.DataFrame(rows)
    path = f"results_{method_slug}.csv"
    df.to_csv(path, index=False)
    print(f"Wrote {path} ({len(df)} rows)")
    return df


## Original ACORN vs. Our Implementation (read this first)

**(A) What ACORN-gamma / ACORN-1 actually do** (Patel, Kraft, Guestrin, Zaharia, 2024): predicate-subgraph traversal on top of HNSW. ACORN-gamma's construction gives each node `M*gamma` candidate edges (metadata-agnostic search over the graph being built); at query time a node's neighbor lookup scans its up-to-`M*gamma` list, keeps only predicate-passing neighbors, and truncates to the first `M`. Optional compression (`M_beta <= M*gamma`) keeps only the nearest `M_beta` candidates explicitly and recovers the rest via two-hop expansion at search time. ACORN-1 is the low-overhead variant: construction is *plain, unmodified HNSW* (gamma=1, M_beta=M), and all neighbor expansion (one-hop + two-hop) happens at search time before filtering/truncating.

**(B/C) What each implementation below reproduces vs. approximates:**
- *ACORN-lite (hnswlib-based)*: an inflated-`M` HNSW index (`M_acorn = M * gamma`) as a stand-in for ACORN's denser predicate-agnostic graph, searched with hnswlib's filter callback (same mechanism as Notebook 05) as a stand-in for predicate-subgraph traversal. **This is not real ACORN** — it never performs neighbor-list truncation to M-per-predicate, never does the two-hop compression-based neighbor lookup, and has no fallback-to-pre-filtering rule.
- *ACORN-style educational reproduction (pure Python/NumPy, small scale)*: directly implements Algorithm 2's neighbor-lookup mechanism (`N^l_p(v)`: scan the raw `M*gamma`-size neighbor list, keep the first `M` predicate-passing candidates), optionally with the two-hop compression-based lookup (`M_beta`). This is a faithful *mechanism* reproduction at small scale (pure-Python graph traversal, slow), meant to answer 'does the idea work at all', not to produce a throughput benchmark.

**(D) vs (E):** neither implementation's numbers are comparable to the paper's own throughput figures (different hardware, different scale, different graph library) — no ACORN paper QPS/recall numbers are copied into this notebook's results CSV, anywhere.

## 9. ACORN-lite (hnswlib-based approximation)

`gamma = 1 / min_selectivity_supported`. With a 5% selectivity floor in this study's filter set, `gamma ~= 20`, so `M_acorn = HNSW_M * gamma`.

In [9]:
ACORN_GAMMA = 20  # 1 / 0.05 floor selectivity supported
M_ACORN = HNSW_M * ACORN_GAMMA
print(f"ACORN-lite: gamma={ACORN_GAMMA}  M_acorn={M_ACORN}")

t0 = time.time()
acorn_index = hnswlib.Index(space=HNSW_SPACE, dim=corpus_emb.shape[1])
acorn_index.init_index(max_elements=len(corpus_emb),
                        ef_construction=HNSW_EF_CONSTRUCTION, M=M_ACORN)
acorn_index.add_items(corpus_emb, np.arange(len(corpus_emb)))
acorn_index.set_ef(HNSW_EF_SEARCH)
acorn_build_time_s = time.time() - t0
print(f"Built ACORN-lite (inflated-M) HNSW in {acorn_build_time_s:.2f}s")


ACORN-lite: gamma=20  M_acorn=320
Built ACORN-lite (inflated-M) HNSW in 2.21s


In [10]:
acorn_lite_rows = []

for filter_name, (target_pct, mask, ftype) in FILTERS.items():
    actual_pct = 100.0 * mask.sum() / len(mask)
    gt_ids, _ = exact_topk_filtered(query_emb, corpus_emb, mask, K)
    mask_bool = mask

    def predicate(label_id):
        return bool(mask_bool[label_id])

    def run():
        lat = []
        ids_all = np.full((len(query_emb), K), -1, dtype=np.int64)
        for qi in range(len(query_emb)):
            t0 = time.perf_counter()
            hits, _ = acorn_index.knn_query(query_emb[qi:qi+1], k=K, filter=predicate)
            lat.append(time.perf_counter() - t0)
            ids_all[qi] = (list(hits[0]) + [-1] * K)[:K]
        run.last_ids = ids_all
        return lat

    lat_stats = time_queries(run)
    recall = recall_at_k(run.last_ids, gt_ids, K)
    acorn_lite_rows.append({
        "method": "acorn_lite_hnswlib", "filter": filter_name,
        "selectivity": round(actual_pct, 3), "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": acorn_build_time_s, "gamma": ACORN_GAMMA,
    })
    print(f"{filter_name:28s} sel={actual_pct:6.2f}%  recall={recall:.3f}")

acorn_lite_df = pd.DataFrame(acorn_lite_rows)
acorn_lite_df


sel100_unfiltered            sel=100.00%  recall=1.000
sel75_label_ne_3             sel= 74.63%  recall=1.000
sel50_label_in_01            sel= 51.17%  recall=1.000
sel25_label_eq_0             sel= 25.60%  recall=1.000
sel25_label_eq_1             sel= 25.57%  recall=1.000
sel25_label_eq_2             sel= 23.47%  recall=1.000
sel25_label_eq_3             sel= 25.37%  recall=1.000
sel10_label0_tag_lt_0.4      sel=  9.73%  recall=1.000
sel5_label0_tag_lt_0.2       sel=  5.20%  recall=1.000


,method,filter,selectivity,recall_at_10,mean_latency_ms,p50_latency_ms,p95_latency_ms,qps,index_build_time_s,gamma
0,acorn_lite_hnswlib,sel100_unfiltered,100.000,1.0,1.151119,0.502202,2.626135,868.719621,2.209084,20
1,acorn_lite_hnswlib,sel75_label_ne_3,74.633,1.0,1.318111,0.620554,3.848715,758.661270,2.209084,20
2,acorn_lite_hnswlib,sel50_label_in_01,51.167,1.0,1.538322,1.082688,3.868895,650.058889,2.209084,20
3,acorn_lite_hnswlib,sel25_label_eq_0,25.600,1.0,1.946505,1.625024,5.096359,513.741327,2.209084,20
4,acorn_lite_hnswlib,sel25_label_eq_1,25.567,1.0,1.592635,1.604111,2.395041,627.890147,2.209084,20
5,acorn_lite_hnswlib,sel25_label_eq_2,23.467,1.0,1.252570,1.266838,2.074850,798.358709,2.209084,20
6,acorn_lite_hnswlib,sel25_label_eq_3,25.367,1.0,1.936589,1.486814,4.514504,516.371883,2.209084,20
7,acorn_lite_hnswlib,sel10_label0_tag_lt_0.4,9.733,1.0,2.583857,2.427013,4.775590,387.018286,2.209084,20
8,acorn_lite_hnswlib,sel5_label0_tag_lt_0.2,5.200,1.0,3.697530,3.246926,7.558362,270.450822,2.209084,20


## 10. ACORN-style educational reproduction (small-scale, pure Python/NumPy)

Runs on a reduced subsample controlled by `ACORN_EDU_N` (independent of `N_CORPUS`). Implements Algorithm 2's core mechanism directly: raw neighbor lists at `M*gamma` size, filter-based neighbor lookup `N^l_p(v)` (first `M` predicate-passing candidates), and the optional compression-based two-hop lookup with `M_beta`.

In [11]:
ACORN_EDU_N = min(2000, len(corpus_emb))  # small-scale, pure-Python is slow
edu_idx = np.random.default_rng(SEED).choice(len(corpus_emb), size=ACORN_EDU_N, replace=False)
edu_vecs = corpus_emb[edu_idx]
edu_mask_full = {name: mask[edu_idx] for name, (_, mask, _) in FILTERS.items()}

M_EDU = HNSW_M
GAMMA_EDU = ACORN_GAMMA
M_GAMMA_EDU = M_EDU * GAMMA_EDU
M_BETA_EDU = min(M_GAMMA_EDU, M_EDU * 4)  # compression parameter (<= M*gamma)

def brute_force_neighbors(vecs, m_gamma):
    """Predicate-agnostic candidate edges: each node's top m_gamma nearest
    neighbors by cosine sim (metadata-agnostic), as ACORN-gamma construction
    would produce via a metadata-agnostic search over the graph being built."""
    sims = vecs @ vecs.T
    np.fill_diagonal(sims, -np.inf)
    order = np.argsort(-sims, axis=1)[:, :m_gamma]
    return order  # (n, m_gamma) neighbor lists, predicate-agnostic

t0 = time.time()
edu_neighbor_lists = brute_force_neighbors(edu_vecs, M_GAMMA_EDU)
edu_build_time_s = time.time() - t0
print(f"Built ACORN-edu raw neighbor lists ({ACORN_EDU_N} nodes, "
      f"M*gamma={M_GAMMA_EDU}) in {edu_build_time_s:.2f}s")


Built ACORN-edu raw neighbor lists (2000 nodes, M*gamma=320) in 0.12s


In [12]:
def acorn_filtered_neighbor_lookup(node, neighbor_lists, pred_mask, M, M_beta,
                                    two_hop=True):
    """N^l_p(v): scan node's raw M*gamma neighbor list, keep first M passing
    the predicate. Optionally recover more via two-hop (compression-based)
    expansion using only the first M_beta explicit neighbors, per Algorithm 2."""
    raw = neighbor_lists[node]
    explicit = raw[:M_beta]
    passing = [n for n in explicit if pred_mask[n]]
    if len(passing) >= M or not two_hop:
        return passing[:M]
    # two-hop expansion: look at neighbors-of-neighbors among the explicit set
    seen = set(passing) | {node}
    for nb in explicit:
        if len(passing) >= M:
            break
        for nb2 in neighbor_lists[nb][:M_beta]:
            if nb2 not in seen and pred_mask[nb2]:
                passing.append(nb2); seen.add(nb2)
                if len(passing) >= M:
                    break
    return passing[:M]

def acorn_edu_search(query_vec, entry_point, neighbor_lists, pred_mask, vecs,
                      k, M, M_beta, max_visits=500):
    """Greedy best-first search restricted to the predicate subgraph induced
    by acorn_filtered_neighbor_lookup — the educational reproduction's
    analogue of ACORN's predicate-subgraph traversal."""
    visited = set()
    frontier = [entry_point]
    best = []
    while frontier and len(visited) < max_visits:
        node = frontier.pop()
        if node in visited:
            continue
        visited.add(node)
        if pred_mask[node]:
            sim = float(query_vec @ vecs[node])
            best.append((sim, node))
        neigh = acorn_filtered_neighbor_lookup(node, neighbor_lists, pred_mask, M, M_beta)
        frontier.extend(n for n in neigh if n not in visited)
    best.sort(key=lambda x: -x[0])
    return [n for _, n in best[:k]]


In [13]:
acorn_edu_rows = []
edu_query_emb = query_emb  # same query set; searched against the small edu subsample

for filter_name in FILTERS:
    pred_mask = edu_mask_full[filter_name]
    valid_pos = np.nonzero(pred_mask)[0]
    if len(valid_pos) == 0:
        continue
    actual_pct = 100.0 * pred_mask.sum() / len(pred_mask)

    gt_ids, _ = exact_topk_filtered(edu_query_emb, edu_vecs, pred_mask, K)

    def run():
        lat = []
        ids_all = np.full((len(edu_query_emb), K), -1, dtype=np.int64)
        for qi in range(len(edu_query_emb)):
            entry = int(valid_pos[0])
            t0 = time.perf_counter()
            found = acorn_edu_search(edu_query_emb[qi], entry, edu_neighbor_lists,
                                      pred_mask, edu_vecs, K, M_EDU, M_BETA_EDU)
            lat.append(time.perf_counter() - t0)
            ids_all[qi] = (found + [-1] * K)[:K]
        run.last_ids = ids_all
        return lat

    lat_stats = time_queries(run)
    recall = recall_at_k(run.last_ids, gt_ids, K)
    acorn_edu_rows.append({
        "method": "acorn_edu_small_scale", "filter": filter_name,
        "selectivity": round(actual_pct, 3), "recall_at_10": recall,
        "mean_latency_ms": lat_stats["mean_latency_ms"],
        "p50_latency_ms": lat_stats["p50_latency_ms"],
        "p95_latency_ms": lat_stats["p95_latency_ms"],
        "qps": lat_stats["qps"],
        "index_build_time_s": edu_build_time_s,
        "acorn_edu_n": ACORN_EDU_N, "M_beta": M_BETA_EDU,
    })
    print(f"{filter_name:28s} sel={actual_pct:6.2f}%  recall={recall:.3f}  "
          f"(edu, n={ACORN_EDU_N})")

acorn_edu_df = pd.DataFrame(acorn_edu_rows)
acorn_edu_df


sel100_unfiltered            sel=100.00%  recall=0.361  (edu, n=2000)
sel75_label_ne_3             sel= 75.50%  recall=0.441  (edu, n=2000)
sel50_label_in_01            sel= 51.20%  recall=0.516  (edu, n=2000)
sel25_label_eq_0             sel= 25.55%  recall=0.981  (edu, n=2000)
sel25_label_eq_1             sel= 25.65%  recall=0.945  (edu, n=2000)
sel25_label_eq_2             sel= 24.30%  recall=0.996  (edu, n=2000)
sel25_label_eq_3             sel= 24.50%  recall=1.000  (edu, n=2000)
sel10_label0_tag_lt_0.4      sel=  9.85%  recall=0.994  (edu, n=2000)
sel5_label0_tag_lt_0.2       sel=  5.25%  recall=1.000  (edu, n=2000)


,method,filter,selectivity,recall_at_10,mean_latency_ms,p50_latency_ms,p95_latency_ms,qps,index_build_time_s,acorn_edu_n,M_beta
0,acorn_edu_small_scale,sel100_unfiltered,100.00,0.361,17.463378,15.716750,26.143356,57.262689,0.115423,2000,64
1,acorn_edu_small_scale,sel75_label_ne_3,75.50,0.441,26.542118,28.521885,41.788482,37.675968,0.115423,2000,64
2,acorn_edu_small_scale,sel50_label_in_01,51.20,0.516,20.298926,15.702535,36.686382,49.263690,0.115423,2000,64
3,acorn_edu_small_scale,sel25_label_eq_0,25.55,0.981,25.570167,23.342169,34.162821,39.108074,0.115423,2000,64
4,acorn_edu_small_scale,sel25_label_eq_1,25.65,0.945,32.469797,31.842920,56.998978,30.797852,0.115423,2000,64
5,acorn_edu_small_scale,sel25_label_eq_2,24.30,0.996,20.855871,18.196018,36.472231,47.948129,0.115423,2000,64
6,acorn_edu_small_scale,sel25_label_eq_3,24.50,1.000,10.040743,9.189640,16.777112,99.594219,0.115423,2000,64
7,acorn_edu_small_scale,sel10_label0_tag_lt_0.4,9.85,0.994,16.551306,16.167449,20.898118,60.418194,0.115423,2000,64
8,acorn_edu_small_scale,sel5_label0_tag_lt_0.2,5.25,1.000,12.968038,14.385033,17.979389,77.112669,0.115423,2000,64


## 11. Save combined results

In [14]:
results_df = pd.concat([acorn_lite_df, acorn_edu_df], ignore_index=True)
results_df.to_csv("results_acorn.csv", index=False)
print(f"Wrote results_acorn.csv ({len(results_df)} rows)")
results_df


Wrote results_acorn.csv (18 rows)


,method,filter,selectivity,recall_at_10,mean_latency_ms,p50_latency_ms,p95_latency_ms,qps,index_build_time_s,gamma,acorn_edu_n,M_beta
0,acorn_lite_hnswlib,sel100_unfiltered,100.000,1.000,1.151119,0.502202,2.626135,868.719621,2.209084,20.0,NaN,NaN
1,acorn_lite_hnswlib,sel75_label_ne_3,74.633,1.000,1.318111,0.620554,3.848715,758.661270,2.209084,20.0,NaN,NaN
2,acorn_lite_hnswlib,sel50_label_in_01,51.167,1.000,1.538322,1.082688,3.868895,650.058889,2.209084,20.0,NaN,NaN
3,acorn_lite_hnswlib,sel25_label_eq_0,25.600,1.000,1.946505,1.625024,5.096359,513.741327,2.209084,20.0,NaN,NaN
4,acorn_lite_hnswlib,sel25_label_eq_1,25.567,1.000,1.592635,1.604111,2.395041,627.890147,2.209084,20.0,NaN,NaN
5,acorn_lite_hnswlib,sel25_label_eq_2,23.467,1.000,1.252570,1.266838,2.074850,798.358709,2.209084,20.0,NaN,NaN
6,acorn_lite_hnswlib,sel25_label_eq_3,25.367,1.000,1.936589,1.486814,4.514504,516.371883,2.209084,20.0,NaN,NaN
7,acorn_lite_hnswlib,sel10_label0_tag_lt_0.4,9.733,1.000,2.583857,2.427013,4.775590,387.018286,2.209084,20.0,NaN,NaN
8,acorn_lite_hnswlib,sel5_label0_tag_lt_0.2,5.200,1.000,3.697530,3.246926,7.558362,270.450822,2.209084,20.0,NaN,NaN
9,acorn_edu_small_scale,sel100_unfiltered,100.000,0.361,17.463378,15.716750,26.143356,57.262689,0.115423,NaN,2000.0,64.0
